In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import test_transforms
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir)
transformed_dataset = ImageDataset(annot_path, img_dir, test_transforms)

In [3]:
def collate_fn(batch):
    # unpacks the batch, and then zip creates separate tuples for images and targets
    images, targets = zip(*batch)

    images = torch.stack(images)
    targets = list(targets)

    return images, targets

In [4]:
from torch.utils.data import DataLoader

dl = DataLoader(transformed_dataset, batch_size=4, collate_fn=collate_fn)
dl

In [5]:
X_batch, y_batch = next(iter(dl))

X_batch.shape, len(y_batch)

(torch.Size([4, 3, 224, 224]), 4)

In [6]:
from src.utilities import pairwise_giou_cxcywh, cxcywh_to_xyxy

bbox_preds = torch.randn(100, 4)
class_preds = torch.randn(100, 21)

In [7]:
_, truth_labels = next(iter(transformed_dataset))
truth_boxes = truth_labels[:, -4:]
truth_boxes.shape, truth_boxes

(torch.Size([5, 4]),
 tensor([[0.5848, 0.7321, 0.1205, 0.3393],
         [0.4196, 0.8482, 0.1741, 0.2902],
         [0.0714, 0.8259, 0.1250, 0.3482],
         [0.5357, 0.6562, 0.1071, 0.2812],
         [0.5893, 0.5402, 0.0714, 0.0893]]))

In [8]:
giou = pairwise_giou_cxcywh(truth_boxes, truth_boxes)

In [9]:
giou

tensor([[ 1.0000, -0.3209, -0.6967,  0.1593, -0.0323],
        [-0.3209,  1.0000, -0.4574, -0.3316, -0.6091],
        [-0.6967, -0.4574,  1.0000, -0.7380, -0.8394],
        [ 0.1593, -0.3316, -0.7380,  1.0000, -0.1367],
        [-0.0323, -0.6091, -0.8394, -0.1367,  1.0000]])

In [15]:
from src.matching import compute_cls_cost, compute_l1_cost, compute_giou_cost

cls_cost = compute_cls_cost(class_preds, truth_labels[:, 0])
l1_cost = compute_l1_cost(bbox_preds, truth_boxes)
giou_cost = compute_giou_cost(bbox_preds, truth_boxes)

cls_cost.shape, l1_cost.shape, giou_cost.shape

(torch.Size([100, 5]), torch.Size([100, 5]), torch.Size([100, 5]))

In [18]:
cls_cost[0], l1_cost[0], giou_cost[0]

(tensor([-0.0222, -0.0079, -0.0511, -0.1699, -0.0249]),
 tensor([1.6496, 1.6049, 1.2433, 1.5643, 1.7643]),
 tensor([0.6380, 0.3352, 0.1236, 0.6439, 0.9374]))

In [19]:
(cls_cost + l1_cost + giou_cost)[0]

tensor([2.2653, 1.9321, 1.3158, 2.0383, 2.6767])

In [20]:
1.0*cls_cost[0], 5.0*l1_cost[0], 2.0*giou_cost[0]

(tensor([-0.0222, -0.0079, -0.0511, -0.1699, -0.0249]),
 tensor([8.2478, 8.0246, 6.2165, 7.8215, 8.8215]),
 tensor([1.2759, 0.6703, 0.2472, 1.2878, 1.8747]))